In [0]:
spark.sql("USE CATALOG med_atlas_ai")
spark.sql("USE SCHEMA default")

In [0]:
%sql

SELECT *
FROM facility_records
WHERE procedures IS NOT NULL AND equipment IS NOT NULL 
LIMIT 5;

In [0]:
%sql

SELECT *
FROM facility_records;

In [0]:
%sql

SELECT *
FROM facility_facts;

In [0]:
%sql

SELECT *
FROM regional_insights 
WHERE total_doctors IS NOT NULL OR total_beds IS NOT NULL;

In [0]:
%sql
    
-- Pick one specific hospital and count how many atomic facts it generated
SELECT 
  r.facility_name,
  r.city,
  COUNT(f.fact_id) as total_atomic_facts_generated,
  -- Let's peek at the first 3 facts the LLM extracted for this hospital
  ARRAY_JOIN(SLICE(COLLECT_LIST(f.fact_text), 1, 5), ' | ') as sample_facts
FROM facility_records r
JOIN facility_facts f 
  ON r.facility_id = f.facility_id 
WHERE f.fact_type != 'regional_insight' -- Exclude our injected summaries
GROUP BY r.facility_name, r.city
ORDER BY total_atomic_facts_generated DESC
LIMIT 5;